# COVER-KBC Profile F1 Full TEST Run

This notebook runs the full blind TEST split once with Profile F1.

It does not evaluate TEST locally. It writes a 475-row `predictions.jsonl` and packages a leaderboard zip.

Important: Colab can only pull code that has been committed and pushed to GitHub. If the Profile F1 changes are still only local, commit/push them before running this notebook.

In [ ]:
# CELL 0 - Runtime and paths
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/vquclinh/FactElicit-AKBC"
REPO = Path("/content/FactElicit-AKBC")
CONFIG_PATH = Path("configs/experiments/cover_kbc_v3_8_profile_f1_stock_empty_rescue_test.yaml")
OUT_DIR = Path("/content/profile_f1_full_test_run")
SUBMISSION_ZIP = Path("/content/profile_f1_full_test_submission.zip")

subprocess.run(["nvidia-smi"], check=False)
print("Python:", sys.version)
print("Output directory:", OUT_DIR)
print("Submission zip:", SUBMISSION_ZIP)

In [ ]:
# CELL 1 - Clone or pull the latest repo code
if REPO.exists():
    os.chdir(REPO)
    subprocess.check_call(["git", "status", "--short"])
    subprocess.check_call(["git", "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])

os.chdir(REPO)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Working directory:", Path.cwd())
print("Git HEAD:", head)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CONFIG_PATH}. Commit/push the Profile F1 code first, then rerun this cell."
    )

In [ ]:
# CELL 2 - Install package and model dependencies
os.chdir(REPO)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf]"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "bitsandbytes",
    "accelerate",
    "huggingface_hub",
    "mistral-common>=1.6.2",
])

import cover_kbc
print("cover_kbc:", cover_kbc.__file__)

In [ ]:
# CELL 3 - Hugging Face login
import getpass

token = os.environ.get("HF_TOKEN")

if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print("Colab secret read failed:", repr(exc))

if not token:
    token = getpass.getpass("Paste HF_TOKEN: ")

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token, add_to_git_credential=False)

print("HF token is available.")

In [ ]:
# CELL 4 - Cheap preflight checks before loading weights
import yaml

os.chdir(REPO)
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

assert cfg["experiment"]["name"] == "cover_kbc_v3_8_profile_f1_stock_empty_rescue_test"
assert cfg["experiment"]["split"] == "test"
assert cfg["experiment"]["frozen_baseline"]["hidden_test_overall_f1"] == 0.5878
assert cfg["leaderboard_repair"]["profile"] == "F1_STOCK_EMPTY_RESCUE"
assert cfg["leaderboard_repair"]["features"]["mistral_stock_empty_rescue"] is True
assert cfg["leaderboard_repair"]["max_calls_by_relation"]["companyTradesAtStockExchange"] == 4

model_profile_text = json.dumps(cfg["model_profile"], sort_keys=True)
assert "Qwen" not in model_profile_text
assert "mistralai/Mistral-Small-3.2-24B-Instruct-2506" in model_profile_text
assert cfg["budget_assertion"]["total_published_parameters"] == 24011361280

subprocess.check_call([sys.executable, "scripts/audit_model_budget.py", str(CONFIG_PATH)])

print("Profile F1 preflight passed.")

In [ ]:
# CELL 5 - Run the full blind TEST split once
os.chdir(REPO)

if OUT_DIR.exists():
    raise FileExistsError(
        f"{OUT_DIR} already exists. Move/delete it manually if you intentionally want a fresh full TEST run."
    )

cmd = [
    sys.executable,
    "scripts/run_cover.py",
    "--config",
    str(CONFIG_PATH),
    "--split",
    "test",
    "--output-dir",
    str(OUT_DIR),
    "--no-eval",
]

print("Running:", " ".join(cmd))
completed = subprocess.run(cmd, text=True)
print("Return code:", completed.returncode)
if completed.returncode != 0:
    raise SystemExit(completed.returncode)

In [ ]:
# CELL 6 - Validate output shape and repair accounting
PREDICTIONS = OUT_DIR / "predictions.jsonl"
MANIFEST = OUT_DIR / "manifest.json"
REPAIR_ACCOUNTING = OUT_DIR / "repair_accounting.json"

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

rows = read_jsonl(PREDICTIONS)
assert len(rows) == 475
assert not (OUT_DIR / "metrics.json").exists(), "TEST is blind; metrics.json should not exist."
assert MANIFEST.exists()
assert REPAIR_ACCOUNTING.exists()

relations = {}
empty_rows = 0
for row in rows:
    assert set(row) == {"SubjectEntity", "Relation", "ObjectEntities"}
    relations[row["Relation"]] = relations.get(row["Relation"], 0) + 1
    if not row["ObjectEntities"]:
        empty_rows += 1

expected_relations = {
    "awardWonBy": 10,
    "companyTradesAtStockExchange": 100,
    "countryLandBordersCountry": 67,
    "hasArea": 100,
    "hasCapacity": 98,
    "personHasCityOfDeath": 100,
}
assert relations == expected_relations, relations

repair = json.loads(REPAIR_ACCOUNTING.read_text(encoding="utf-8"))
stock = repair["by_relation"]["companyTradesAtStockExchange"]["stock_empty_rescue"]

print("Predictions:", PREDICTIONS)
print("Rows:", len(rows))
print("Relations:", relations)
print("Empty rows:", empty_rows)
print("Repair profile:", repair["profile"])
print("Total repair calls:", repair["total_repair_calls"])
print("Stock empty rescue:")
print(json.dumps(stock, indent=2, sort_keys=True))

In [ ]:
# CELL 7 - Package the TEST submission zip
os.chdir(REPO)

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

subprocess.check_call([
    sys.executable,
    "scripts/package_submission.py",
    "--predictions",
    str(PREDICTIONS),
    "--input",
    "benchmark/data/test.jsonl",
    "--split",
    "test",
    "--out",
    str(SUBMISSION_ZIP),
])

print("Submit this zip:", SUBMISSION_ZIP)
print("Raw predictions:", PREDICTIONS)

In [ ]:
# CELL 8 - Download artifacts
from google.colab import files

files.download(str(SUBMISSION_ZIP))
files.download(str(PREDICTIONS))
files.download(str(MANIFEST))
files.download(str(REPAIR_ACCOUNTING))